In [1]:
import os
import ast
import json 
import getpass
import logging
from pathlib import Path
from time import sleep
import subprocess
import sys

from langchain_core.tools import Tool
from langgraph.graph import StateGraph, END
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain.tools import BaseTool

from typing import TypedDict, Dict, List

from PyDI.io import load_xml, load_parquet, load_csv
from PyDI.profiling import DataProfiler

/Users/nickblazek/Desktop/Agent_Pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Initialize

In [2]:
OUTPUT_DIR = "output/"
INPUT_DIR = "input/"

INCLUDE_DOCS = False # IMPORTANT: Use carefully since token usage is increased drastically with documentation
USE_LLM = "gpt"
#USE_LLM = "gemini"
#USE_LLM = "groq"

In [3]:
# Not supported anymore
#USE_LLM = "gemini_broken"

In [4]:
if USE_LLM == "gemini": # or USE_LLM == "gemini_broken":
    if "GOOGLE_API_KEY" not in os.environ:
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")
elif USE_LLM == "groq":
    if "GROQ_API_KEY" not in os.environ:
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")
elif USE_LLM == "gpt":
    if "OPENAI_API_KEY" not in os.environ:
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAi API key: ")

In [5]:
logging.basicConfig(filename= OUTPUT_DIR + 'agent.log',
                    filemode='a',
                    format='%(asctime)s,%(msecs)d %(name)s %(levelname)s %(message)s',
                    datefmt='%H:%M:%S',
                    level=logging.DEBUG,
                    encoding='utf-8')

## Utilities

In [6]:
def load_dataset(path):
    # check file exists
    if not os.path.exists(path):
        raise FileNotFoundError(f"Dataset not found at: {path}")
    ext = os.path.splitext(path)[1].lower()

    # load dataset according to extension
    if ext == ".parquet":
        df = load_parquet(path)
    elif ext == ".csv":
        df = load_csv(path)
    elif ext == ".xml":
        df = load_xml(path, nested_handling="aggregate")
    else:
        raise ValueError(f"Unsupported format: {ext}. Supported: .csv, .parquet, .xml")
    return df
    

## Tools

In [7]:
class ProfileDatasetTool(BaseTool):
    name: str = "profile_dataset"
    description: str = """
        A tool that takes the path as a argument called `path` of type str of the dataset file as string and performs data analysis. 
        A JSON string is returned with the profile data.
    """

#    if USE_LLM == "gemini_broken":
#        def _run(self, *args, **kwargs) -> str:
#            
#            # Accept path either positionally or as a kwarg (e.g. "__arg1")
#            path = None
#            if args:
#                # first positional arg
#                path = args[0]
#            elif kwargs:
#                # sometimes the LLM -> function call maps the single arg to "__arg1"
#                # or some other autogenerated key. If there's exactly one kwarg, take it.
#                if len(kwargs) == 1:
#                    path = list(kwargs.values())[0]
#                else:
#                    # if multiple kwargs provided, try to find a param named "path"
#                    path = kwargs.get("path") or kwargs.get("file_path") or None
#
#            return self.create_profile(path)
#            
#    elif USE_LLM == "groq" or USE_LLM == "gpt" or USE_LLM == "gemini":
    
    def _run(self, path) -> str:   
        return self.create_profile(path)

    def create_profile(self, path):

        if not path or not isinstance(path, str):
                raise ValueError("ProfileDatasetTool requires a single path string argument.")

        df = load_dataset(path)    
        
        if df is None or getattr(df, "empty", False):
            # return a structured JSON error string (LLM will see this as content)
            return json.dumps({"error": f"Dataset at {path} loaded as empty or failed to load."})

        profiler = DataProfiler()
        profile = profiler.summary(df, print_summary=False)

        # ensure to return a JSON string (your docstring promised JSON string)
        try:
            profile_json = json.dumps(profile, default=str)
        except Exception:
            # fallback: convert to str if not json-serializable
            profile_json = json.dumps({"profile_str": str(profile)})

        return profile_json
        

## Blocking Tester (Standalone)
Run this cell **BEFORE** the main agent pipeline to:
1. Evaluate blocking strategies using LLM reasoning
2. Auto-detect ID columns for each dataset
3. Find optimal blocking configurations (columns, strategy type)
4. Save the config to `blocking_config.json`

The output `blocking_config` can then be passed to the main `SimpleModelAgent` to guide pipeline generation.

In [8]:
import pandas as pd
import re
import traceback
from typing import Dict, List, Tuple, Any, Optional
from PyDI.entitymatching import StandardBlocker, EmbeddingBlocker, TokenBlocker, SortedNeighbourhoodBlocker, EntityMatchingEvaluator

# Strategy descriptions with computational guidance for LLM
STRATEGY_DESCRIPTIONS = """
Available blocking strategies (ranked by computational cost):

1. **exact_match_single** - Block on exact match of ONE column
   - Speed: ⚡ FASTEST
   - Parameters: columns (1 column only)
   - Best for: Highly standardized values (IDs, codes, normalized names)
   - Candidate pairs: Low (only exact matches)

2. **exact_match_multi** - Block on exact match of MULTIPLE columns combined
   - Speed: ⚡ FAST
   - Parameters: columns (2+ columns)
   - Best for: When single column has too many collisions
   - Candidate pairs: Very low (more restrictive)

3. **sorted_neighbourhood** - Sliding window over sorted records
   - Speed: ⚡⚡ FAST-MEDIUM (O(n log n) sorting + linear scan)
   - Parameters: columns (1 column = sort key), window (INTEGER 5-50, default 15)
   - Best for: Data with natural ordering (dates, years, alphabetically sortable names)
   - WINDOW SIZE HEURISTIC: Use window ≈ max(10, min(50, sqrt(N_left + N_right) / 10))
   - For ~20K total rows: window=10-20 is reasonable
   - For ~100K total rows: window=20-35 is reasonable
   - Candidate pairs: Approximately (N_left + N_right) × window / 2
   - WARNING: Only effective if sort key has meaningful ordering for matching entities

4. **token_blocking** - Block on shared word tokens
   - Speed: ⚡⚡ MEDIUM
   - Parameters: columns (1 column), min_token_len (INTEGER 3-10, default 4)
   - Best for: Text fields with meaningful words (titles, names)
   - WARNING: min_token_len=2 creates HUGE blocks (common words like "the", "a", "of")
   - RECOMMENDATION: Use min_token_len >= 4 for large datasets (>10K rows)
   - Candidate pairs: Can be HIGH if min_token_len too small

5. **ngram_blocking** - Block on character n-grams (substrings)
   - Speed: ⚡⚡⚡ MEDIUM-SLOW
   - Parameters: columns (1 column), ngram_size (INTEGER 3-6, default 4)
   - Best for: Fuzzy matching, typos, abbreviations, non-English text
   - WARNING: Small ngram_size (2-3) creates MASSIVE candidate sets
   - RECOMMENDATION: Use ngram_size >= 4 for datasets > 5K rows
   - Candidate pairs: Can be VERY HIGH if ngram_size too small

6. **semantic_similarity** - Embedding-based similarity blocking
   - Speed: ⚡⚡⚡⚡ SLOWEST (requires embedding computation)
   - Parameters: columns (1+ columns), top_k (INTEGER 10-100, default 20)
   - Best for: Semantic variations, different wording for same entity
   - Candidate pairs: Controlled by top_k (predictable: left_size × top_k)

COMPUTATIONAL GUIDELINES:
- For datasets with N_left × N_right > 100M potential pairs: Use exact_match or token_blocking with min_token_len >= 5
- For datasets with N_left × N_right > 10M potential pairs: Avoid ngram_size < 4 or min_token_len < 4
- Target candidate pairs: Ideally < 2M for reasonable runtime
- Pair Completeness (PC) goal: >= 0.80 (find 80% of true matches)
"""

class BlockingTester:
    """LLM-driven blocking tester that evaluates strategies with full parameter control."""
    
    def __init__(
        self, 
        llm,
        datasets: List[str],
        blocking_testsets: Dict[Tuple[str, str], str],
        output_dir: str = "output/blocking-evaluation",
        pc_threshold: float = 0.80,
        max_attempts: int = 3,
        max_error_retries: int = 2,
        max_candidates: int = 2_000_000,
        candidate_tolerance: float = 0.15,
        verbose: bool = True
    ):
        self.llm = llm
        self.output_dir = output_dir
        self.pc_threshold = pc_threshold
        self.max_attempts = max_attempts
        self.max_error_retries = max_error_retries
        self.max_candidates = max_candidates
        self.candidate_tolerance = candidate_tolerance
        self.verbose = verbose
        self.evaluator = EntityMatchingEvaluator()
        self.results_history: List[Dict[str, Any]] = []
        
        os.makedirs(self.output_dir, exist_ok=True)
        
        # Load datasets
        self.datasets_loaded: Dict[str, pd.DataFrame] = {}
        for path in datasets:
            name = os.path.splitext(os.path.basename(path))[0]
            if self.verbose:
                print(f"[*] Loading dataset: {name}")
            self.datasets_loaded[name] = load_dataset(path)
        
        # Load gold standards
        self.gold_standards: Dict[Tuple[str, str], pd.DataFrame] = {}
        for pair, path in blocking_testsets.items():
            if self.verbose:
                print(f"[*] Loading gold standard for {pair[0]} <-> {pair[1]}")
            self.gold_standards[pair] = self._load_gold_standard(path)
    
    def _load_gold_standard(self, path: str) -> pd.DataFrame:
        """Load and prepare gold standard CSV."""
        gs = pd.read_csv(path)
        col_mapping = {}
        for col in gs.columns:
            col_lower = col.lower()
            if 'id_a' in col_lower or col_lower == 'id1':
                col_mapping[col] = 'id1'
            elif 'id_b' in col_lower or col_lower == 'id2':
                col_mapping[col] = 'id2'
            elif 'label' in col_lower:
                col_mapping[col] = 'label'
        gs = gs.rename(columns=col_mapping)
        if 'label' in gs.columns:
            gs = gs[gs['label'] == 1.0].copy()
        if self.verbose:
            print(f"    Loaded {len(gs)} ground truth pairs")
        return gs[['id1', 'id2']]
    
    def _detect_id_column(self, df: pd.DataFrame) -> str:
        """Simple heuristic to detect ID column."""
        for col in df.columns:
            col_lower = col.lower()
            if col_lower in ['id', 'record_id', 'row_id', 'index', 'key']:
                return col
            if 'id' in col_lower:
                try:
                    if df[col].nunique() == len(df):
                        return col
                except TypeError:
                    pass
        return df.columns[0]
    
    def _analyze_columns_for_pair(self, name_left: str, name_right: str, id_column: str = None) -> Dict[str, Any]:
        """Analyze columns for a dataset pair with computational estimates."""
        df_left = self.datasets_loaded[name_left]
        df_right = self.datasets_loaded[name_right]
        
        left_size = len(df_left)
        right_size = len(df_right)
        potential_pairs = left_size * right_size
        
        common_cols = set(df_left.columns) & set(df_right.columns)
        if id_column:
            common_cols.discard(id_column)
        common_cols.discard('id')
        
        column_details = {}
        for col in common_cols:
            left_col, right_col = df_left[col], df_right[col]
            try:
                sample_left = [str(x)[:50] for x in left_col.dropna().head(2).tolist()]
                sample_right = [str(x)[:50] for x in right_col.dropna().head(2).tolist()]
                avg_tokens = 0
                if left_col.dtype == 'object':
                    avg_tokens = left_col.dropna().head(100).apply(lambda x: len(str(x).split())).mean()
            except:
                sample_left, sample_right = [], []
                avg_tokens = 0
            
            try:
                unique_left = int(left_col.nunique())
            except TypeError:
                unique_left = -1
            try:
                unique_right = int(right_col.nunique())
            except TypeError:
                unique_right = -1
            
            column_details[col] = {
                "dtype": str(left_col.dtype),
                "null_pct": f"{left_col.isnull().mean()*100:.0f}%/{right_col.isnull().mean()*100:.0f}%",
                "unique_left": unique_left,
                "unique_right": unique_right,
                "avg_tokens": round(avg_tokens, 1),
                "samples": sample_left[:1] + sample_right[:1]
            }
        
        return {
            "left_dataset": name_left,
            "right_dataset": name_right,
            "left_size": left_size,
            "right_size": right_size,
            "potential_pairs": potential_pairs,
            "common_columns": list(common_cols),
            "column_details": column_details
        }
    
    def _ask_llm_for_strategy(self, analysis: Dict[str, Any], previous_attempts: List[Dict] = None, last_error: str = None) -> Dict[str, Any]:
        """Ask LLM to select a blocking strategy with full parameter control."""
        
        system_prompt = f"""You are a blocking strategy optimizer for entity matching.

{STRATEGY_DESCRIPTIONS}

Given dataset characteristics, select the BEST blocking strategy.

RESPOND WITH ONLY A JSON OBJECT:
{{
    "strategy": "one of: exact_match_single, exact_match_multi, sorted_neighbourhood, token_blocking, ngram_blocking, semantic_similarity",
    "columns": ["column_name"],
    "min_token_len": 4,
    "ngram_size": 4,
    "window": 15,
    "top_k": 20,
    "reasoning": "brief explanation of choice and parameter values"
}}

RULES:
- strategy MUST be one of: exact_match_single, exact_match_multi, sorted_neighbourhood, token_blocking, ngram_blocking, semantic_similarity
- columns MUST exist in common_columns
- For exact_match_single: use exactly 1 column
- For exact_match_multi: use 2+ columns
- For sorted_neighbourhood: use 1 column with natural ordering (dates, years, sortable names)
- For token_blocking/ngram_blocking: only first column is used
- min_token_len: INTEGER between 3-10 (only for token_blocking)
- ngram_size: INTEGER between 3-6 (only for ngram_blocking)
- window: INTEGER between 5-50 (only for sorted_neighbourhood)
- top_k: INTEGER between 10-100 (only for semantic_similarity)"""

        human_content = f"""Dataset pair: {analysis['left_dataset']} <-> {analysis['right_dataset']}

DATASET SIZES (IMPORTANT FOR COMPUTATIONAL DECISIONS):
- Left dataset: {analysis['left_size']:,} rows
- Right dataset: {analysis['right_size']:,} rows  
- Potential pairs (N×M): {analysis['potential_pairs']:,}
- Max allowed candidates: {self.max_candidates:,}

Common columns: {analysis['common_columns']}

Column details:
{json.dumps(analysis['column_details'], indent=2)}"""
        
        if last_error:
            human_content += f"""

⚠️ PREVIOUS STRATEGY FAILED WITH ERROR:
{last_error[:500]}

Please fix the strategy - ensure correct parameter names and valid column references."""
        
        elif previous_attempts:
            attempts_summary = []
            for a in previous_attempts:
                status = "❌ ERROR" if a.get('error') else f"PC={a.get('pair_completeness',0):.3f}"
                candidates = a.get('num_candidates', 'N/A')
                attempts_summary.append(
                    f"  - {a['strategy']}(cols={a['columns']}, params={a.get('params',{})}) → {status}, candidates={candidates}"
                )
            
            human_content += f"""

PREVIOUS ATTEMPTS (all failed to meet PC >= {self.pc_threshold} or had errors):
{chr(10).join(attempts_summary)}

IMPORTANT: Try a DIFFERENT strategy or significantly different parameters.
- If token_blocking failed with too many candidates, increase min_token_len
- If PC was too low, try a less restrictive strategy or different column
- Consider semantic_similarity for fuzzy matching if exact strategies fail"""
        
        response = self.llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=human_content)
        ])
        response_text = response.content if hasattr(response, 'content') else str(response)
        
        return self._parse_llm_response(response_text, analysis['common_columns'])
    
    def _parse_llm_response(self, response_text: str, valid_columns: List[str]) -> Dict[str, Any]:
        """Parse and validate LLM response with all parameters."""
        try:
            cleaned = re.sub(r'^```(?:json)?\n?|```$', '', response_text.strip(), flags=re.MULTILINE)
            cleaned = cleaned.strip()
            parsed = json.loads(cleaned)
        except json.JSONDecodeError:
            match = re.search(r'\{[^{}]*\}', response_text, re.DOTALL)
            if match:
                try:
                    parsed = json.loads(match.group())
                except:
                    parsed = {}
            else:
                parsed = {}
        
        # Validate strategy
        valid_strategies = ['exact_match_single', 'exact_match_multi', 'sorted_neighbourhood', 'token_blocking', 'ngram_blocking', 'semantic_similarity']
        strategy = parsed.get('strategy', 'exact_match_single')
        if strategy not in valid_strategies:
            strategy = 'exact_match_single'
        
        # Validate columns
        columns = parsed.get('columns', [])
        if not columns or not isinstance(columns, list):
            columns = [valid_columns[0]] if valid_columns else ['name']
        columns = [c for c in columns if c in valid_columns]
        if not columns:
            columns = [valid_columns[0]] if valid_columns else ['name']
        
        # Validate parameters with bounds
        min_token_len = parsed.get('min_token_len', 4)
        if not isinstance(min_token_len, int):
            try:
                min_token_len = int(min_token_len)
            except:
                min_token_len = 4
        min_token_len = max(2, min(10, min_token_len))
        
        ngram_size = parsed.get('ngram_size', 4)
        if not isinstance(ngram_size, int):
            try:
                ngram_size = int(ngram_size)
            except:
                ngram_size = 4
        ngram_size = max(2, min(6, ngram_size))
        
        top_k = parsed.get('top_k', 20)
        if not isinstance(top_k, int):
            try:
                top_k = int(top_k)
            except:
                top_k = 20
        top_k = max(5, min(100, top_k))
        
        window = parsed.get('window', 15)
        if not isinstance(window, int):
            try:
                window = int(window)
            except:
                window = 15
        window = max(3, min(100, window))
        
        return {
            'strategy': strategy,
            'columns': columns,
            'min_token_len': min_token_len,
            'ngram_size': ngram_size,
            'window': window,
            'top_k': top_k,
            'reasoning': parsed.get('reasoning', 'No reasoning provided')
        }
    
    def _strategy_to_config(self, choice: Dict[str, Any]) -> Dict[str, Any]:
        """Convert strategy choice to blocker config."""
        strategy, columns = choice['strategy'], choice['columns']
        
        if strategy == 'exact_match_single':
            return {'name': f"Standard ({columns[0]})", 'type': 'standard', 'on': [columns[0]]}
        elif strategy == 'exact_match_multi':
            return {'name': f"Standard ({'+'.join(columns)})", 'type': 'standard', 'on': columns}
        elif strategy == 'sorted_neighbourhood':
            return {'name': f"SortedNeighbourhood ({columns[0]}, w={choice['window']})", 'type': 'sorted_neighbourhood',
                    'key': columns[0], 'window': choice['window']}
        elif strategy == 'semantic_similarity':
            return {'name': f"Embedding ({'+'.join(columns)}, k={choice['top_k']})", 'type': 'embedding', 
                    'text_cols': columns, 'top_k': choice['top_k']}
        elif strategy == 'token_blocking':
            return {'name': f"Token ({columns[0]}, len>={choice['min_token_len']})", 'type': 'token', 
                    'column': columns[0], 'min_token_len': choice['min_token_len']}
        elif strategy == 'ngram_blocking':
            return {'name': f"NGram ({columns[0]}, n={choice['ngram_size']})", 'type': 'ngram', 
                    'column': columns[0], 'ngram_size': choice['ngram_size']}
        
        return {'name': f"Standard ({columns[0]})", 'type': 'standard', 'on': [columns[0]]}
    
    def create_blocker(self, name_left: str, name_right: str, blocker_type: str, id_column: str = "id", **kwargs):
        """Create a blocker instance."""
        df_left, df_right = self.datasets_loaded[name_left], self.datasets_loaded[name_right]
        common_params = {'output_dir': self.output_dir, 'id_column': id_column}
        
        if blocker_type == 'standard':
            return StandardBlocker(df_left, df_right, on=kwargs.get('on', []), batch_size=1000, **common_params)
        elif blocker_type == 'sorted_neighbourhood':
            return SortedNeighbourhoodBlocker(df_left, df_right, key=kwargs.get('key', 'name'),
                                              window=kwargs.get('window', 15), batch_size=1000, **common_params)
        elif blocker_type == 'embedding':
            return EmbeddingBlocker(df_left, df_right, text_cols=kwargs.get('text_cols', []), 
                                   top_k=kwargs.get('top_k', 20), batch_size=1000, **common_params)
        elif blocker_type == 'token':
            return TokenBlocker(df_left, df_right, column=kwargs.get('column', 'name'), 
                               min_token_len=kwargs.get('min_token_len', 4), batch_size=1000, **common_params)
        elif blocker_type == 'ngram':
            return TokenBlocker(df_left, df_right, column=kwargs.get('column', 'name'),
                               ngram_size=kwargs.get('ngram_size', 4), ngram_type="character",
                               batch_size=1000, **common_params)
        
        raise ValueError(f"Unknown blocker type: {blocker_type}")
    
    def _is_acceptable(self, pc: float, num_candidates: int) -> Tuple[bool, str]:
        """Check if result is acceptable, with tolerance for excellent PC."""
        hard_limit = self.max_candidates
        soft_limit = int(self.max_candidates * (1 + self.candidate_tolerance))
        
        if pc >= self.pc_threshold and num_candidates <= hard_limit:
            return True, "perfect"
        
        if pc >= self.pc_threshold + 0.10 and num_candidates <= soft_limit:
            return True, "acceptable_with_tolerance"
        
        if num_candidates > soft_limit:
            return False, "exceeds_hard_limit"
        elif num_candidates > hard_limit:
            return False, "exceeds_soft_limit"
        else:
            return False, "low_pc"
    
    def evaluate_blocker(self, blocker, gold_standard: pd.DataFrame, name: str = "blocker") -> Dict[str, Any]:
        """Evaluate a blocker against ground truth."""
        candidates = blocker.materialize()
        num_candidates = len(candidates)
        
        if self.verbose:
            print(f"    {name}: {num_candidates:,} candidates")
        
        hard_limit = int(self.max_candidates * (1 + self.candidate_tolerance))
        
        if num_candidates > hard_limit:
            if self.verbose:
                print(f"    ⛔ SKIPPING EVALUATION: {num_candidates:,} > {hard_limit:,} (hard limit)")
                print(f"    💡 Try increasing min_token_len or ngram_size, or use a more selective column")
            return {
                'method': name,
                'num_candidates': num_candidates,
                'num_gold_pairs': len(gold_standard),
                'pair_completeness': 0,
                'reduction_ratio': 0,
                'is_acceptable': False,
                'acceptance_reason': 'exceeds_hard_limit',
                'skipped_evaluation': True
            }
        
        result = self.evaluator.evaluate_blocking(
            candidate_pairs=candidates, 
            blocker=blocker, 
            test_pairs=gold_standard, 
            out_dir=self.output_dir
        )
        pc = result.get('pair_completeness', 0)
        is_acceptable, reason = self._is_acceptable(pc, num_candidates)
        
        result.update({
            'method': name, 
            'num_candidates': num_candidates, 
            'num_gold_pairs': len(gold_standard),
            'is_acceptable': is_acceptable,
            'acceptance_reason': reason,
            'skipped_evaluation': False
        })
        
        if self.verbose:
            rr = result.get('reduction_ratio', 0)
            if is_acceptable:
                status = "✅" if reason == "perfect" else "✅ (within tolerance)"
            else:
                status = "⚠️"
            print(f"    {status} PC={pc:.4f}, RR={rr:.4f}")
        
        return result
    
    def _try_execute_strategy(self, strategy_choice: Dict, name_left: str, name_right: str, 
                              gold: pd.DataFrame, id_column: str) -> Tuple[Optional[Dict], Optional[str]]:
        """Try to execute a blocking strategy."""
        config = self._strategy_to_config(strategy_choice)
        blocker_type = config.pop('type')
        blocker_name = config.pop('name')
        
        try:
            blocker = self.create_blocker(name_left, name_right, blocker_type=blocker_type, 
                                         id_column=id_column, **config)
            result = self.evaluate_blocker(blocker, gold, name=blocker_name)
            result.update({
                'pair': f"{name_left}_{name_right}", 
                'strategy': strategy_choice['strategy'],
                'columns': strategy_choice['columns'], 
                'reasoning': strategy_choice['reasoning'],
                'params': {
                    'min_token_len': strategy_choice.get('min_token_len'),
                    'ngram_size': strategy_choice.get('ngram_size'),
                    'window': strategy_choice.get('window'),
                    'top_k': strategy_choice.get('top_k')
                }
            })
            return result, None
        except Exception as e:
            error_msg = f"{type(e).__name__}: {str(e)}\n{traceback.format_exc()[-500:]}"
            return None, error_msg
    
    def run_pair_with_llm(self, name_left: str, name_right: str, id_column: str = None) -> Dict[str, Any]:
        """Run LLM-driven blocking evaluation for a dataset pair."""
        pair_key = (name_left, name_right)
        reverse_key = (name_right, name_left)
        
        if pair_key in self.gold_standards:
            gold = self.gold_standards[pair_key]
        elif reverse_key in self.gold_standards:
            gold = self.gold_standards[reverse_key]
        else:
            raise ValueError(f"No gold standard found for pair: {pair_key}")
        
        soft_limit = int(self.max_candidates * (1 + self.candidate_tolerance))
        print(f"\n{'='*60}")
        print(f"🤖 BLOCKING: {name_left} <-> {name_right}")
        print(f"{'='*60}")
        print(f"Gold pairs: {len(gold)}, PC threshold: {self.pc_threshold}")
        print(f"Candidate limit: {self.max_candidates:,} (hard: {soft_limit:,} with tolerance)")
        
        if id_column is None:
            id_col_left = self._detect_id_column(self.datasets_loaded[name_left])
            id_col_right = self._detect_id_column(self.datasets_loaded[name_right])
            
            if id_col_left != id_col_right:
                print(f"    ID columns differ ({id_col_left}/{id_col_right}), renaming to 'record_id'")
                self.datasets_loaded[name_left] = self.datasets_loaded[name_left].rename(columns={id_col_left: 'record_id'})
                self.datasets_loaded[name_right] = self.datasets_loaded[name_right].rename(columns={id_col_right: 'record_id'})
                id_column = 'record_id'
            else:
                id_column = id_col_left
            print(f"    Using ID column: '{id_column}'")
        
        analysis = self._analyze_columns_for_pair(name_left, name_right, id_column)
        print(f"Dataset sizes: {analysis['left_size']:,} × {analysis['right_size']:,} = {analysis['potential_pairs']:,} potential pairs")
        print(f"Common columns: {analysis['common_columns']}")
        
        previous_attempts = []
        best_result = None
        
        for attempt in range(self.max_attempts):
            print(f"\n--- Attempt {attempt + 1}/{self.max_attempts} ---")
            
            strategy_choice = self._ask_llm_for_strategy(analysis, previous_attempts if attempt > 0 else None)
            print(f"LLM chose: {strategy_choice['strategy']} on {strategy_choice['columns']}")
            print(f"  Params: min_token_len={strategy_choice['min_token_len']}, ngram_size={strategy_choice['ngram_size']}, window={strategy_choice['window']}, top_k={strategy_choice['top_k']}")
            print(f"  Reasoning: {strategy_choice['reasoning'][:100]}...")
            
            error_retries = 0
            last_error = None
            result = None
            
            while error_retries <= self.max_error_retries:
                if error_retries > 0:
                    print(f"  Retry {error_retries}/{self.max_error_retries} after error...")
                    strategy_choice = self._ask_llm_for_strategy(analysis, previous_attempts, last_error=last_error)
                    print(f"  LLM retry chose: {strategy_choice['strategy']} on {strategy_choice['columns']}")
                
                result, error = self._try_execute_strategy(strategy_choice, name_left, name_right, gold, id_column)
                
                if error:
                    print(f"  ❌ Error: {error[:200]}...")
                    last_error = error
                    error_retries += 1
                    continue
                
                break
            
            if result is None:
                previous_attempts.append({
                    'strategy': strategy_choice['strategy'],
                    'columns': strategy_choice['columns'],
                    'params': {
                        'min_token_len': strategy_choice.get('min_token_len'),
                        'ngram_size': strategy_choice.get('ngram_size'),
                        'window': strategy_choice.get('window'),
                        'top_k': strategy_choice.get('top_k')
                    },
                    'error': True
                })
                continue
            
            previous_attempts.append({
                'strategy': strategy_choice['strategy'],
                'columns': strategy_choice['columns'],
                'params': {
                    'min_token_len': strategy_choice.get('min_token_len'),
                    'ngram_size': strategy_choice.get('ngram_size'),
                    'window': strategy_choice.get('window'),
                    'top_k': strategy_choice.get('top_k')
                },
                'pair_completeness': result.get('pair_completeness', 0),
                'num_candidates': result.get('num_candidates', 0),
                'error': False
            })
            
            if result.get('is_acceptable', False):
                print(f"✅ Found acceptable strategy!")
                best_result = result
                break
            
            if best_result is None or result.get('pair_completeness', 0) > best_result.get('pair_completeness', 0):
                best_result = result
        
        if best_result is None:
            print(f"⚠️ No successful strategy found after {self.max_attempts} attempts")
            best_result = {
                'pair': f"{name_left}_{name_right}",
                'strategy': 'failed',
                'columns': [],
                'pair_completeness': 0,
                'num_candidates': 0,
                'is_acceptable': False
            }
        
        self.results_history.append(best_result)
        return best_result
    
    def run_all_pairs(self) -> Tuple[pd.DataFrame, Dict[str, Any]]:
        """Run blocking evaluation for all dataset pairs."""
        all_results = []
        discovered_config = {
            "blocking_strategies": {},
            "id_columns": {}
        }
        
        for (name_left, name_right) in self.gold_standards.keys():
            result = self.run_pair_with_llm(name_left, name_right)
            all_results.append(result)
            
            if result.get('is_acceptable', False):
                discovered_config["blocking_strategies"][f"{name_left}_{name_right}"] = {
                    "strategy": result['strategy'],
                    "columns": result['columns'],
                    "params": result.get('params', {}),
                    "pair_completeness": result.get('pair_completeness', 0),
                    "num_candidates": result.get('num_candidates', 0)
                }
        
        for name, df in self.datasets_loaded.items():
            for col in df.columns:
                if 'id' in col.lower():
                    discovered_config["id_columns"][name] = col
                    break
        
        df = pd.DataFrame(all_results)
        
        if not df.empty:
            df.to_csv(os.path.join(self.output_dir, "blocking_tester_results.csv"), index=False)
            print(f"\n{'='*60}")
            print(f"📊 RESULTS SUMMARY: {len(df)} pairs evaluated")
            print(f"{'='*60}")
            for _, row in df.iterrows():
                pc = row.get('pair_completeness', 0)
                candidates = row.get('num_candidates', 0)
                is_acceptable = row.get('is_acceptable', False)
                status = "✅" if is_acceptable else "⚠️"
                print(f"  {status} {row['pair']}: PC={pc:.4f}, candidates={candidates:,}, strategy={row['strategy']}")
        
        with open(os.path.join(self.output_dir, "blocking_config.json"), "w") as f:
            json.dump(discovered_config, f, indent=2)
        print(f"\n💾 Config saved to: {self.output_dir}/blocking_config.json")
        
        return df, discovered_config

In [9]:
if USE_LLM == "gemini":
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash",
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
    )
elif USE_LLM == "groq":
    llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0.2
    )
elif USE_LLM == "gpt":
    llm = ChatOpenAI(
        model="gpt-5.1",
        temperature=0,
        max_tokens=None,
    )
# Define blocking gold standard testsets
blocking_testsets = {
    ("discogs", "lastfm"): INPUT_DIR + "testsets/music/discogs_lastfm_goldstandard_blocking.csv",
    ("discogs", "musicbrainz"): INPUT_DIR + "testsets/music/discogs_musicbrainz_goldstandard_blocking.csv",
    ("musicbrainz", "lastfm"): INPUT_DIR + "testsets/music/musicbrainz_lastfm_goldstandard_blocking.csv",
}
datasets = [
    INPUT_DIR + "datasets/discogs.xml",
    INPUT_DIR + "datasets/lastfm.xml",
    INPUT_DIR + "datasets/musicbrainz.xml"
]
# Initialize LLM-driven blocking tester (uses same llm as main pipeline)
tester = BlockingTester(
    llm=llm,  # Pass the same LLM instance
    datasets=datasets,
    blocking_testsets=blocking_testsets,
    output_dir="output/blocking-evaluation",
    pc_threshold=0.80,       # Retry if pair_completeness below this
    max_attempts=3,          # Max strategy attempts per dataset pair
    max_error_retries=2,     # Max error fix retries per strategy
    verbose=True
)

# Run LLM-driven evaluation for all pairs
# Returns: (results_df, blocking_config)

blocking_results, blocking_config = tester.run_all_pairs()
print("Pass 'blocking_config' to your pipeline to use the optimized blocking settings.")
print("This config is saved to: output/blocking-evaluation/blocking_config.json")

# The blocking_config can now be passed to the main agent or saved for later use
print("="*60)
print("\n" + "="*60)
print("🔧 BLOCKING CONFIG READY FOR MAIN AGENT")

[*] Loading dataset: discogs
[*] Loading dataset: lastfm
[*] Loading dataset: musicbrainz
[*] Loading gold standard for discogs <-> lastfm
    Loaded 55 ground truth pairs
[*] Loading gold standard for discogs <-> musicbrainz
    Loaded 78 ground truth pairs
[*] Loading gold standard for musicbrainz <-> lastfm
    Loaded 79 ground truth pairs

🤖 BLOCKING: discogs <-> lastfm
Gold pairs: 55, PC threshold: 0.8
Candidate limit: 2,000,000 (hard: 2,300,000 with tolerance)
    Using ID column: 'id'
Dataset sizes: 22,627 × 9,865 = 223,215,355 potential pairs
Common columns: ['artist', 'tracks_track_position', 'name', 'tracks_track_name', 'duration', 'tracks_track_duration']

--- Attempt 1/3 ---
LLM chose: token_blocking on ['name']
  Params: min_token_len=5, ngram_size=4, window=15, top_k=20
  Reasoning: With ~223M potential pairs and a 2M candidate cap, we need a strong but not overly strict blocker on...
    Token (name, len>=5): 2,176,544 candidates
    ✅ (within tolerance) PC=0.9818, RR=0.

In [13]:
# Restaurant use case datasets
restaurant_datasets = [
    INPUT_DIR + "datasets/kaggle_small.parquet",
    INPUT_DIR + "datasets/uber_eats_small.parquet",
    INPUT_DIR + "datasets/yelp_small.parquet"
]

# Define blocking gold standard testsets for restaurants
restaurant_blocking_testsets = {
    ("kaggle_small", "uber_eats_small"): INPUT_DIR + "testsets/music/Restaurants_Sample_Test_Stes/kaggle_uber_blocking_goldstandard.csv",
    ("kaggle_small", "yelp_small"): INPUT_DIR + "testsets/music/Restaurants_Sample_Test_Stes/kaggle_yelp_blocking_goldstandard.csv",
    ("yelp_small", "uber_eats_small"): INPUT_DIR + "testsets/music/Restaurants_Sample_Test_Stes/yelp_uber_blocking_goldstandard.csv",
}

# Initialize LLM-driven blocking tester for restaurants
restaurant_tester = BlockingTester(
    llm=llm,                         # Pass the same LLM instance
    datasets=restaurant_datasets,
    blocking_testsets=restaurant_blocking_testsets,
    output_dir="output/blocking-evaluation-restaurants",
    pc_threshold=0.80,               # Retry if pair_completeness below this
    max_attempts=3,                  # Max strategy attempts per dataset pair
    max_error_retries=2,             # Max error fix retries per strategy
    max_candidates=2_000_000,        # Maximum allowed candidate pairs
    candidate_tolerance=0.15,        # Tolerance for candidate count overflow
    verbose=True
)

# Run LLM-driven evaluation for all restaurant pairs
# Returns: (results_df, blocking_config)
restaurant_blocking_results, restaurant_blocking_config = restaurant_tester.run_all_pairs()

print("Pass 'restaurant_blocking_config' to your pipeline to use the optimized blocking settings.")
print("This config is saved to: output/blocking-evaluation-restaurants/blocking_config.json")

# The blocking_config can now be passed to the main agent or saved for later use
print("=" * 60)
print("\n" + "=" * 60)
print("🔧 RESTAURANT BLOCKING CONFIG READY FOR MAIN AGENT")

[*] Loading dataset: kaggle_small
[*] Loading dataset: uber_eats_small
[*] Loading dataset: yelp_small
[*] Loading gold standard for kaggle_small <-> uber_eats_small
    Loaded 58 ground truth pairs
[*] Loading gold standard for kaggle_small <-> yelp_small
    Loaded 40 ground truth pairs
[*] Loading gold standard for yelp_small <-> uber_eats_small
    Loaded 22 ground truth pairs

🤖 BLOCKING: kaggle_small <-> uber_eats_small
Gold pairs: 58, PC threshold: 0.8
Candidate limit: 2,000,000 (hard: 2,300,000 with tolerance)
    ID columns differ (kaggle380k_id/uber_eats_id), renaming to 'record_id'
    Using ID column: 'record_id'
Dataset sizes: 10,000 × 10,000 = 100,000,000 potential pairs
Common columns: ['state', 'source', 'longitude', 'street', 'city', 'rating', 'name', 'latitude', 'categories', 'postal_code', 'country', 'name_norm', 'address_line1', 'address_line2', 'house_number']

--- Attempt 1/3 ---
LLM chose: exact_match_multi on ['postal_code', 'house_number']
  Params: min_token_l

## Agents

In [11]:
class SimpleModelAgentState(TypedDict):
    datasets: list
    fusion_testset: str
    blocking_config: Dict  # NEW: Blocking config from BlockingTester
    
    data_profiles: Dict
    
    integration_pipeline_code: str
    pipeline_execution_result: str
    pipeline_execution_attempts: int

    evaluation_code: str
    evaluation_execution_result: str
    evaluation_execution_attempts: int

    evaluation_attempts: int
    evaluation_metrics: Dict

class SimpleModelAgent:
    
    def __init__(self, model, profileTool):
        # initialize logger
        self.logger = logging.getLogger()
        
        # prepare the StateGraph
        graph = StateGraph(SimpleModelAgentState)

        # create nodes
        graph.add_node("profile_data", self.profile_data)
        graph.add_node("pipeline_adaption", self.pipeline_adaption)
        graph.add_node("execute_pipeline", self.execute_pipeline)
        graph.add_node("evaluation_adaption", self.evaluation_adaption)
        graph.add_node("execute_evaluation", self.execute_evaluation)
        graph.add_node("evaluation_decision", self.evaluation_decision)

        # create edges
        graph.add_edge("profile_data", "pipeline_adaption")
        graph.add_edge("pipeline_adaption", "execute_pipeline")
        
        graph.add_conditional_edges(
            "execute_pipeline",
            lambda state: (
                "evaluation_adaption"
                if isinstance(state.get("pipeline_execution_result", ""), str)
                   and state["pipeline_execution_result"].lower().startswith("success")
                else "pipeline_adaption"
                if state.get("pipeline_execution_attempts", 0) < 3
                else END
            ),
            {
                "pipeline_adaption": "pipeline_adaption",
                "evaluation_adaption": "evaluation_adaption",
                END: END
            }
        )

        graph.add_edge("evaluation_adaption", "execute_evaluation")

        graph.add_conditional_edges(
            "execute_evaluation",
            lambda state: (
                "evaluation_decision" 
                if isinstance(state.get("evaluation_execution_result", ""), str)
                   and state["evaluation_execution_result"].lower().startswith("success")
                else "evaluation_adaption"
                if state.get("evaluation_attempts", 0) < 3
                else END
            ),
            {
                "evaluation_adaption": "evaluation_adaption",
                "evaluation_decision": "evaluation_decision",
                END: END
            }
        )

        graph.add_conditional_edges(
            "evaluation_decision",
            lambda state: (
                "pipeline_adaption"
                if state.get("evaluation_metrics", {}).get("overall_accuracy", 0) < 0.85
                   and state.get("evaluation_attempts", 0) < 3
                else END
            ),
            {
                "pipeline_adaption": "pipeline_adaption",
                END: END
            }
        )

        graph.set_entry_point("profile_data")
        self.graph = graph.compile()

        # make tools available to the model based on model type
        self.tools = {profileTool.name: profileTool}
#        if USE_LLM == "groq" or USE_LLM == "gpt" or USE_LLM == "gemini":
        self.model = model.bind_tools([profileTool])
#        elif USE_LLM == "gemini_broken":
#            self.model = model.bind_tools(profileTool)
        
    # Creates tool calls to profile the data and saves it into agent state 
    def profile_data(self, state:SimpleModelAgentState):
        self.logger.info('----------------------- Entering profile_data -----------------------')

        print("[*] Profiling datasets")

        system_prompt = """
            You are a data scientist tasked with the integration of several datasets.
            For each dataset path provided, call the tool `profile_dataset` with the path
            (one tool call per dataset).
        """
        
        datasets_list_str = "\n".join(state['datasets'])
        human_content = f"Please profile these datasets (one call per dataset):\n{datasets_list_str}"
        message = [SystemMessage(content=system_prompt), HumanMessage(content=human_content)]
        self.logger.info("Input Message:" + str(message))
        
        result = self.model.invoke(message)
        self.logger.info("RESULT:" + str(result))

        # call tools
        tool_calls = result.tool_calls

        self.logger.info("Tool Calls:" + str(tool_calls))
        results = {}
        for t in tool_calls:
            if not t['name'] in self.tools:      # check for bad tool name from LLM
                self.logger.info("adapt_pipeline: ....bad tool name....")
                result = "bad tool name, retry"  # instruct LLM to retry if bad
            else:
                result = self.tools[t['name']].invoke(t['args'])
            
#            if USE_LLM == "groq" or USE_LLM == "gpt" or USE_LLM == "gemini":
            results[t['args']['path']] = result
#            elif USE_LLM == "gemini_broken":
#                results[t['args']['__arg1']] = result

        with open(OUTPUT_DIR + "profile/profiles.json", "w") as file:
            file.write(json.dumps(results, indent=2))
          
        self.logger.info('Leaving profile_data')
        return {'data_profiles': results}

    def pipeline_adaption(self, state: SimpleModelAgentState):
        self.logger.info('----------------------- Entering pipeline_adaption -----------------------')

        ####### PREPARE SYSTEM PROMT #######
    
        # Load example pipeline code
        example_pipeline_path = os.path.join(INPUT_DIR, "example_pipelines/example_pipeline.py")
        if not os.path.exists(example_pipeline_path):
            raise FileNotFoundError(f"Example pipeline not found at: {example_pipeline_path}")
    
        with open(example_pipeline_path, "r", encoding="utf-8") as f:
            example_pipeline_code = f.read()

        # Prepare first-row previews for each dataset
        dataset_previews = {}
        for path in state['datasets']:
            df = load_dataset(path)
            if df is not None and not df.empty:
                # Take only first row and convert to dictionary
                dataset_previews[path] = df.iloc[0].to_dict()
            else:
                dataset_previews[path] = "Failed to load or empty dataset"
                
        # Prepare prompt for the model
        system_prompt = f"""
        You are a data scientist tasked with the integration of several datasets.
        You are provided with the following inputs:
    
        1. An example integration pipeline (Python code using PyDI library). Pay
        attention to the comments within the pipeline, as they also contain important instructions:
        {example_pipeline_code}
    
        2. A list of dataset file locations:
        {json.dumps(state['datasets'], indent=2)}
    
        3. The first row of each dataset to help understand the structure:
        {json.dumps(dataset_previews, indent=2)}
    
        4. A dictionary containing the profile data of the datasets
        (including number of rows, nulls_per_column and dtypes of 
        the columns):
        {json.dumps(state['data_profiles'], indent=2)}
        """
        
        # Include blocking config if available (from BlockingTester)
        blocking_config = state.get('blocking_config')
        if blocking_config:
            system_prompt += f"""

        5. **BLOCKING CONFIGURATION** (pre-computed optimal blocking strategies):
        This configuration was determined by a blocking evaluation agent. Use these settings
        for your blocking step in the pipeline:
        {json.dumps(blocking_config, indent=2)}
        
        IMPORTANT: Use the id_columns and blocking_strategies from this config:
        - Use the correct id_column for each dataset as specified
        - Use the recommended strategy (exact_match_single/multi or semantic_similarity)
        - Use the recommended columns for blocking
        """

        system_prompt += """

        Your task is to **create a similar integration pipeline** so that it works with
        the datasets provided. Output should only consist of the relevant Python code
        for the integration pipeline.
        """

        # Include documentation if global variable is set
        if INCLUDE_DOCS:
            doc_path = os.path.join(INPUT_DIR, "documentation")
            if os.path.exists(doc_path):
                doc_files = [f for f in os.listdir(doc_path) if f.endswith(".md")]
                all_docs = ""
                for f_name in doc_files:
                    with open(os.path.join(doc_path, f_name), "r", encoding="utf-8", errors="replace") as f:
                        all_docs += f.read() + "\n\n"
                system_prompt += "\n\n### Documentation\n" + all_docs

        ####### PREPARE TASK PROMT #######

        # Determine if this is initial generation, fix due to execution error, or fix due to poor evaluation
        if "pipeline_execution_result" not in state and "evaluation_metrics" not in state:
            print("[*] Creating initial pipeline")
            human_content = "Create the integration pipeline for the provided datasets."
    
        else:
            # Either a pipeline execution error or evaluation-based adaption
            broken_pipeline_path = OUTPUT_DIR + "code/pipeline.py"
            with open(broken_pipeline_path, "r", encoding="utf-8", errors="ignore") as f:
                broken_code = f.read()
    
            human_content = "You previously generated Python integration pipeline code.\n"
            
            # Add execution error if exists
            if "pipeline_execution_result" in state and state["pipeline_execution_result"].lower().startswith("error"):
                print("[*] Trying to fix pipeline")
                human_content += f"Executing this pipeline caused the following error:\n{state['pipeline_execution_result']}\n"
    
            # Add evaluation feedback if available
            if "evaluation_metrics" in state:
                print("[*] Trying to improve pipeline based on evaluation")
                human_content += f"Evaluation of the last run shows the following metrics:\n{json.dumps(state['evaluation_metrics'], indent=2)}\n"
                human_content += "Improve the pipeline to increase overall accuracy and correct errors highlighted by the evaluation.\n"
    
            human_content += "Here is the current pipeline code:\n" + broken_code
            human_content += "\nOutput ONLY the corrected Python code."
            
        ####### EXECUTE PROMT #######
        
        message = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=human_content)
        ]
    
        self.logger.info("Input Message:" + str(message))
    
        # Call the model
        adapted_pipeline = self.model.invoke(message)
        adapted_pipeline = adapted_pipeline.content if hasattr(adapted_pipeline, "content") else str(adapted_pipeline)

        self.logger.info("RESULT:" + str(adapted_pipeline))
        
        # Strip leading/trailing ``` if present
        if adapted_pipeline.startswith("```python") and adapted_pipeline.endswith("```"):
            adapted_pipeline = adapted_pipeline[9:-3].strip()
        
        with open(OUTPUT_DIR + "code/pipeline.py", 'w', errors="ignore") as file:
            file.write(str(adapted_pipeline))
    
        self.logger.info('Leaving pipeline_adaption')
        return {'integration_pipeline_code': adapted_pipeline}

    def execute_pipeline(self, state: SimpleModelAgentState):
        self.logger.info('----------------------- Entering execute_pipeline -----------------------')

        print("[*] Running generated pipeline")

        attempts = state.get("pipeline_execution_attempts", 0) + 1
        state["pipeline_execution_attempts"] = attempts
    
        pipeline_path = OUTPUT_DIR + "code/pipeline.py"
    
        try:
            result = subprocess.run(
                [sys.executable, pipeline_path],
                capture_output=True,
                text=True,
                timeout=3600
            )
    
            if result.returncode == 0:
                print("[+] Pipeline successfully completed")
                execution_result = "success"
            else:
                print("[-] Pipeline could not be executed")
                execution_result = f"error: {result.stderr}"
    
        except Exception as e:
            execution_result = f"error: {str(e)}"
    
        self.logger.info("Pipeline execution result: " + execution_result)
        self.logger.info('Leaving execute_pipeline')
    
        return {
            "pipeline_execution_result": execution_result,
            "pipeline_execution_attempts": attempts
        }
        
    def evaluation_adaption(self, state: SimpleModelAgentState):
        self.logger.info('----------------------- Entering evaluation_adaption -----------------------')
        
        example_eval_path = INPUT_DIR + "example_pipelines/example_evaluation.py"
        example_eval_code = open(example_eval_path).read()
    
        fused_output_path = "output/data_fusion/fusion_rb_standard_blocker.csv"
    
        system_prompt = f"""
        You are a data scientist evaluating a data fusion pipeline.
    
        Example evaluation code:
        {example_eval_code}
    
        Generated integration pipeline:
        {state['integration_pipeline_code']}
    
        Dataset profiles:
        {json.dumps(state['data_profiles'], indent=2)}
    
        The fused output is located at:
        {fused_output_path}

        The testset is located at:
        {state['fusion_testset']}
    
        Create evaluation code that:
        - Uses the correct fusion strategy
        - Loads the fused output
        - Loads the gold standard test set
        - Prints structured evaluation metrics
        """

        ####### HUMAN PROMPT #######
        if not state.get("evaluation_execution_result"):
            print("[*] Adapting evaluation code")
            # First generation
            human_content = """
            Create the evaluation code.
            """
        else:
            # Fix broken evaluation
            print("[*] Fixing evaluation code")
            attempts = state.get("evaluation_execution_attempts", 0)
    
            if attempts >= 3:
                self.logger.info("Maximum evaluation fix attempts reached")
                return {"evaluation_execution_result": "error: max attempts reached"}
    
            eval_path = OUTPUT_DIR + "code/evaluation.py"
            with open(eval_path, "r", encoding="utf-8") as f:
                broken_code = f.read()
    
            error = state["evaluation_execution_result"]
    
            human_content = f"""
            You previously generated Python evaluation code.
            Executing this code caused the following error:
    
            {error}
    
            Here is the current evaluation code:
            {broken_code}
    
            Fix the code so that it executes successfully.
            Output ONLY the corrected Python code.
            """
    
        ####### MODEL CALL #######
    
        message = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=human_content)
        ]
    
        self.logger.info("Input Message:" + str(message))
    
        result = self.model.invoke(message)
        code = result.content.strip("```python").strip("```")
    
        self.logger.info("RESULT:" + str(code))
    
        with open(OUTPUT_DIR + "code/evaluation.py", "w") as f:
            f.write(code)

        self.logger.info('Leaving evaluation_adaption')
    
        return {"evaluation_code": code}

    def execute_evaluation(self, state: SimpleModelAgentState):
        self.logger.info('----------------------- Entering execute_evaluation -----------------------')
    
        print("[*] Running generated evaluation")
    
        attempts = state.get("evaluation_execution_attempts", 0) + 1
        state["evaluation_execution_attempts"] = attempts
    
        evaluation_path = OUTPUT_DIR + "code/evaluation.py"
    
        try:
            result = subprocess.run(
                [sys.executable, evaluation_path],
                capture_output=True,
                text=True,
                timeout=3600
            )
    
            if result.returncode == 0:
                print("[+] Evaluation successfully completed")
                execution_result = "success"
            else:
                print("[-] Evaluation could not be executed")
                execution_result = f"error: {result.stderr}"
    
        except Exception as e:
            execution_result = f"error: {str(e)}"
    
        self.logger.info("Evaluation execution result: " + execution_result)
        self.logger.info('Leaving execute_evaluation')
    
        return {
            "evaluation_execution_result": execution_result,
            "evaluation_execution_attempts": attempts
        }

    def evaluation_decision(self, state: SimpleModelAgentState):
        self.logger.info('----------------------- Entering evaluation_decision -----------------------')
        print("[*] Reading evaluation metrics")

        attempts = state.get("evaluation_attempts", 0) + 1
    
        eval_path = "output/pipeline_evaluation/pipeline_evaluation.json"
    
        if not os.path.exists(eval_path):
            self.logger.warning("Evaluation file missing")
            metrics = {"error": "evaluation file missing"}
        else:
            with open(eval_path, "r", encoding="utf-8") as f:
                metrics = json.load(f)
    
        self.logger.info(f"Evaluation metrics: {metrics}")
    
        # Store metrics in the state
        state["evaluation_metrics"] = metrics

        overall_acc = metrics.get("overall_accuracy", 0.0)
        print(f"[*] Overall Accuracy: {overall_acc:.3%}")
    
        self.logger.info('Leaving evaluation_decision')

        return {
            "evaluation_metrics": metrics,
            "evaluation_attempts": attempts,
            "pipeline_execution_result": "",
            "pipeline_execution_attempts": 0,
            "evaluation_execution_result": "",
            "evaluation_execution_attempts": 0
        }


## Invoke Pipeline

In [12]:
# Initialize model
if USE_LLM == "gemini":
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash",
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
    )
elif USE_LLM == "groq":
    llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0.2
    )
elif USE_LLM == "gpt":
    llm = ChatOpenAI(
        model="gpt-5.1",
        temperature=0,
        max_tokens=None,
    )

# input datasets
#datasets = [
#    INPUT_DIR + "datasets/kaggle_small.parquet",
#    INPUT_DIR + "datasets/uber_eats_small.parquet",
#    INPUT_DIR + "datasets/yelp_small.parquet"
#]

#datasets = [
#    INPUT_DIR + "datasets/amazon_small.parquet",
#    INPUT_DIR + "datasets/goodreads_small.parquet",
#    INPUT_DIR + "datasets/metabooks_small.parquet"
#]

datasets = [
    INPUT_DIR + "datasets/discogs.xml",
    INPUT_DIR + "datasets/lastfm.xml",
    INPUT_DIR + "datasets/musicbrainz.xml"
]
fusion_testset = INPUT_DIR + "testsets/music/test_set.xml"

# prepare agent
from workflow_logging import attach_logging
agent = SimpleModelAgent(llm, ProfileDatasetTool())
attach_logging(agent, output_dir=OUTPUT_DIR, summary_model_name="gpt-4.1-mini", notebook_name="AdaptationPipeline_blockingextension_update")


# Load blocking config if available (from BlockingTester run)
# This allows the agent to use pre-computed optimal blocking strategies
blocking_config_path = "output/blocking-evaluation/blocking_config.json"
if os.path.exists(blocking_config_path):
    with open(blocking_config_path, "r") as f:
        blocking_config = json.load(f)
    print(f"✅ Loaded blocking config from: {blocking_config_path}")
    print(f"   ID columns: {list(blocking_config.get('id_columns', {}).keys())}")
    print(f"   Strategies: {list(blocking_config.get('blocking_strategies', {}).keys())}")
else:
    blocking_config = {}
    print("⚠️ No blocking config found. Run BlockingTester first for optimal blocking settings.")
    print("   The agent will generate blocking configuration from scratch.")

# invoke agent with blocking config (if available)
result = agent.graph.invoke({
    "datasets": datasets, 
    "fusion_testset": fusion_testset, 
    "blocking_config": blocking_config
})

✅ Loaded blocking config from: output/blocking-evaluation/blocking_config.json
   ID columns: ['discogs', 'lastfm', 'musicbrainz']
   Strategies: ['discogs_lastfm', 'discogs_musicbrainz', 'musicbrainz_lastfm']
[*] Profiling datasets
[*] Creating initial pipeline


KeyboardInterrupt: 